# MMM benchmark: PyMC-Marketing, DeepCausalMMM (DCM), Google Meridian, Meta Robyn (one notebook)

**Single data path (dashboard):** `examples/dashboard_rmse_optimized.py` → `load_real_mmm_data`, then `UnifiedDataPipeline.temporal_split` and `SimpleGlobalScaler` (`examples/pymc_aligned_dcm_config.json`).

**Models**
- **PyMC-Marketing** — pooled geo MMM on the **full DMA panel** by default (`MAX_DMAS=None` uses every `dmacode` in the CSV; expect long runtimes).
- **DeepCausalMMM (DCM)** — same `train_data` / `holdout_data` as this notebook; **`UnifiedDataPipeline.fit_and_transform_training` → `transform_holdout`** + **`ModelTrainer`** with `pymc_aligned_dcm_config.json`, matching **`dashboard_rmse_optimized.py`**. Summary **R² / RMSE / MAE** use sklearn on **original-scale** `y` after **`inverse_transform_predictions(..., remove_padding=True)`** (strips **`burn_in_weeks`**), so **holdout RMSE is in the same KPI units** as PyMC/Ridge; the PyMC long-DataFrame feature path still differs.
- **Meridian** — same panel / split; smoke MCMC by default.
- **Robyn-style national table** — panel aggregated to **national weekly**; **sklearn Ridge** on `dep_var` for apples-to-apples sklearn **R² / RMSE / MAE** vs the tail holdout. Optional **`robynpy` + R + glmnet** adds Robyn-native metrics if installed.

**Metrics**
- **sklearn** `r2_score`, `sqrt(mean_squared_error)`, `mean_absolute_error` on **original-scale** KPI where we have point predictions (PyMC-Marketing `predict` mean; **DCM** after pipeline inverse transform; national Ridge).
- **Meridian** — `Analyzer.predictive_accuracy` → **R_Squared**, **MAPE**, **wMAPE** for **geo / Test** (vendor definitions; not identical to sklearn). The **`holdout_r2`** column repeats **geo / Test** R² so the summary table aligns with PyMC/sklearn rows (`meridian_R2_geo_test` is the same value).

**Why you should distrust very high R² (especially pooled)**
- **`train_r2`** is fit on the **same** rows the model sees — values **0.95+** are common with rich seasonality + controls; they are **not** a clean “accuracy” claim.
- **Pooled `holdout_r2`** stacks **all geo×week** holdout points and uses **one** global mean in the R² denominator. Shared **time** patterns (seasonality, macro) can make pooled R² **look excellent** even when **per-DMA** fit is mediocre.
- **Trust more:** **`holdout_r2_median_by_geo`** / **`holdout_r2_mean_by_geo`** (R² computed **inside each DMA** on holdout weeks, then aggregated), **`train_minus_holdout_r2`** (overfit gap), **`holdout_rmse` vs `holdout_rmse_naive_mean`**, and **`holdout_mape_pct`**.
- **PyMC** still does **not** put holdout `y` in the likelihood; skepticism is about **metric choice**, not a proven pipeline bug.
- **Meridian** Train vs Test R² are **vendor-defined**; compare Test to PyMC holdout metrics cautiously.

**Dates:** Panel `date` is tied to each CSV **`weekid`** (sorted). If the file has a date-like column (`date`, `week_start`, …), we use the **min date per `weekid`**; otherwise `data_week_anchor` in `pymc_aligned_dcm_config.json` is the calendar date for **`min(weekid)`**, and other weeks add 7-day steps.

**Pipeline audit (this notebook):** After load + `temporal_split`, we **assert** (1) CSV `weekid` count matches tensor time dim, (2) concatenating train then holdout **`y`** exactly reconstructs the pre-split **`y`** (no gap/overlap), (3) `SimpleGlobalScaler`’s stored **`y_mean_per_region`** equals the mean of **training** `y` only (so holdout targets never enter normalization).

**Caveats:** National Robyn-track rows are **one series per week**; DCM/PyMC/Meridian are **geo×week** panels — **do not** rank models by raw R² across scopes. Same **calendar** holdout fraction everywhere (`holdout_ratio` in config).

**Requirements:** `torch`, `deepcausalmmm`, `scikit-learn`, `pandas`, `numpy`. Optional: `pymc-marketing`, `google-meridian` (Python ≥3.10), `robynpy`, R/glmnet.


## Configuration


In [8]:
import json
import os
import sys
from pathlib import Path


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        if (d / "pyproject.toml").is_file() and (d / "deepcausalmmm").is_dir():
            return d
    raise FileNotFoundError("Run from deepcausalmmm repo root or examples/.")


REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_cfg = REPO_ROOT / "examples" / "pymc_aligned_dcm_config.json"
with open(_cfg, encoding="utf-8") as f:
    DCM_CONFIG = json.load(f)

DATA_PATH = "examples/data/MMM Data.csv"
if not os.path.isfile(DATA_PATH):
    DATA_PATH = "data/MMM Data.csv"

# None = all DMAs in the CSV (full panel). Set to int e.g. 25 for faster PyMC/Meridian smoke tests.
MAX_DMAS = None

HOLDOUT_RATIO = float(DCM_CONFIG.get("holdout_ratio", 0.12))
geo_col = "dmacode"

# PyMC MCMC (smoke)
MCMC_PROFILE = "smoke"
if MCMC_PROFILE == "smoke":
    CHAINS, DRAW, TUNE, TARGET_ACCEPT = 1, 150, 400, 0.90
else:
    CHAINS, DRAW, TUNE, TARGET_ACCEPT = 2, 300, 800, 0.92
RANDOM_SEED = 42

# Meridian MCMC (smoke)
N_CHAINS, N_ADAPT, N_BURNIN, N_KEEP = 1, 150, 100, 50

# Optional: set True to try robynpy train_models (needs R + glmnet; slow)
RUN_ROBYN_GLMNET = False
ROBYN_ITERATIONS = 100  # lower for demo

RUN_PYMC = True
RUN_MERIDIAN = True

# DeepCausalMMM: same UnifiedDataPipeline + ModelTrainer path as dashboard_rmse_optimized.py
RUN_DCM = True
# Paper-style: set DCM_N_EPOCHS=1500, DCM_EARLY_PATIENCE=300 to match get_default_config() / dashboard.
DCM_N_EPOCHS = 300
DCM_EARLY_PATIENCE = 80

_dma_disp = "all" if MAX_DMAS is None else str(MAX_DMAS)
print(
    f"REPO_ROOT={REPO_ROOT} | MAX_DMAS={_dma_disp} | holdout_ratio={HOLDOUT_RATIO} | "
    f"RUN_DCM={RUN_DCM} RUN_PYMC={RUN_PYMC} RUN_MERIDIAN={RUN_MERIDIAN} RUN_ROBYN_GLMNET={RUN_ROBYN_GLMNET}"
)


REPO_ROOT=/Users/adityapu/Documents/GitHub/deepcausalmmm | MAX_DMAS=all | holdout_ratio=0.12 | RUN_PYMC=True RUN_MERIDIAN=True RUN_ROBYN_GLMNET=False


## Imports


In [9]:
import importlib.util
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from deepcausalmmm.core.data import UnifiedDataPipeline
from deepcausalmmm.core.scaling import SimpleGlobalScaler

warnings.filterwarnings("ignore", category=UserWarning)


## Shared pipeline: `load_real_mmm_data` + split + PyMC / Meridian / DCM tables


In [10]:
# --- Dashboard loader (same file as dashboard_rmse_optimized.py) ---
from pathlib import Path

_dash_py = REPO_ROOT / "examples" / "dashboard_rmse_optimized.py"
_spec = importlib.util.spec_from_file_location("dashboard_rmse_optimized", _dash_py)
_dash = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_dash)

_data_abs = Path(DATA_PATH)
if not _data_abs.is_file():
    _data_abs = REPO_ROOT / DATA_PATH
if not _data_abs.is_file():
    _data_abs = REPO_ROOT / "data" / "MMM Data.csv"
if not _data_abs.is_file():
    raise FileNotFoundError(DATA_PATH)

X_media, X_control, y, media_names, control_names = _dash.load_real_mmm_data(str(_data_abs))

hdr = pd.read_csv(_data_abs, nrows=0)
impression_cols = [c for c in hdr.columns if "impressions_" in c]
if "target_visits" in hdr.columns:
    target_col = "target_visits"
    value_cols = [c for c in hdr.columns if c.startswith("control_")]
else:
    target_col = "value_visits_visits"
    value_cols = [c for c in hdr.columns if "value_" in c and c != "value_visits_visits"]

regions = sorted(pd.read_csv(_data_abs, usecols=["dmacode"])["dmacode"].unique().tolist())
if MAX_DMAS is not None:
    regions = regions[: int(MAX_DMAS)]
    k = len(regions)
    X_media, X_control, y = X_media[:k], X_control[:k], y[:k]

n_weeks = int(X_media.shape[1])
pipeline = UnifiedDataPipeline(DCM_CONFIG)
train_data, holdout_data = pipeline.temporal_split(X_media, X_control, y)
train_weeks = int(train_data["X_media"].shape[1])
holdout_weeks = int(holdout_data["X_media"].shape[1])

# --- Pipeline safety checks (temporal split + no accidental slice bug) ---
_weeks_csv = sorted(pd.read_csv(_data_abs, usecols=["weekid"])["weekid"].dropna().unique().tolist())
assert len(_weeks_csv) == n_weeks, (
    f"week count mismatch: tensor n_weeks={n_weeks} vs CSV unique weekid={len(_weeks_csv)}"
)
assert train_weeks + holdout_weeks == n_weeks
_y_cat = np.concatenate([train_data["y"], holdout_data["y"]], axis=1)
np.testing.assert_allclose(_y_cat, y, rtol=0, atol=0, err_msg="train|holdout y must reconstruct full y")
assert train_data["X_media"].shape[1] == train_weeks and holdout_data["X_media"].shape[1] == holdout_weeks

scaler = SimpleGlobalScaler(DCM_CONFIG)
scaler.fit(train_data["X_media"], train_data["X_control"], train_data["y"])
_ym_tr = np.asarray(train_data["y"], dtype=np.float64).mean(axis=1)
_ym_stored = np.asarray(scaler.scaling_constants["y_mean_per_region"]).reshape(-1)
np.testing.assert_allclose(_ym_stored, _ym_tr, rtol=1e-4, atol=1e-3)
_, X_ctrl_train_t, _ = scaler.transform(
    train_data["X_media"], train_data["X_control"], train_data["y"]
)
_, X_ctrl_hold_t, _ = scaler.transform(
    holdout_data["X_media"], holdout_data["X_control"], holdout_data["y"]
)
X_ctrl_train = X_ctrl_train_t.numpy()
X_ctrl_hold = X_ctrl_hold_t.numpy()

channel_short = [f"ch{i:02d}" for i in range(1, len(impression_cols) + 1)]
control_cols = list(value_cols)

# Calendar dates from data: map each CSV `weekid` -> Monday (or anchor weekday) + (weekid - min_weekid) weeks.
# If the file adds a real date column (e.g. week_start), we use the min date per weekid instead.
_date_like = [
    c
    for c in hdr.columns
    if c.lower() in ("date", "week_start", "week_date", "period_start")
    or c.lower().endswith("_date")
]
_w0 = int(_weeks_csv[0])
if _date_like:
    _dc = _date_like[0]
    _tmp = pd.read_csv(_data_abs, usecols=["weekid", _dc])
    _tmp[_dc] = pd.to_datetime(_tmp[_dc], errors="coerce")
    _per_w = _tmp.groupby("weekid", sort=True)[_dc].min()
    dates_all = [pd.Timestamp(_per_w.loc[int(w)]) for w in _weeks_csv]
    print(f"Dates from column `{_dc}` (per weekid min)")
else:
    _anchor = pd.Timestamp(DCM_CONFIG.get("data_week_anchor", "2020-01-06"))
    dates_all = [_anchor + pd.Timedelta(weeks=int(w) - _w0) for w in _weeks_csv]
    print(
        f"Dates from weekid + data_week_anchor={_anchor.date()} (weekid min={_w0}); "
        f"set pymc_aligned_dcm_config.json 'data_week_anchor' if week zero should differ."
    )

rows_tr, rows_ho = [], []
for ri, code in enumerate(regions):
    for ti in range(train_weeks):
        row = {geo_col: code, "date": dates_all[ti]}
        for c in range(len(channel_short)):
            row[channel_short[c]] = float(train_data["X_media"][ri, ti, c])
        for c in range(len(control_cols)):
            row[control_cols[c]] = float(X_ctrl_train[ri, ti, c])
        row[target_col] = float(train_data["y"][ri, ti])
        rows_tr.append(row)
    for ti in range(holdout_weeks):
        gti = train_weeks + ti
        row = {geo_col: code, "date": dates_all[gti]}
        for c in range(len(channel_short)):
            row[channel_short[c]] = float(holdout_data["X_media"][ri, ti, c])
        for c in range(len(control_cols)):
            row[control_cols[c]] = float(X_ctrl_hold[ri, ti, c])
        row[target_col] = float(holdout_data["y"][ri, ti])
        rows_ho.append(row)

df_train = pd.DataFrame(rows_tr).sort_values([geo_col, "date"]).reset_index(drop=True)
df_holdout = pd.DataFrame(rows_ho).sort_values([geo_col, "date"]).reset_index(drop=True)
X_train = df_train.drop(columns=[target_col])
y_train = df_train[target_col].copy()
y_train.name = target_col
X_holdout = df_holdout.drop(columns=[target_col])
y_holdout = df_holdout[target_col].copy()

# Meridian long copy
df_panel = pd.concat([df_train, df_holdout], ignore_index=True)
mer_df = df_panel.copy()
mer_df["geo"] = mer_df[geo_col].astype(str)
mer_df["time"] = pd.to_datetime(mer_df["date"]).dt.strftime("%Y-%m-%d")
mer_df["kpi"] = mer_df[target_col].astype(float)
mer_df["revenue_per_kpi"] = 1.0
mer_df["population"] = 1.0e6
spend_cols = []
for ch in channel_short:
    sc = f"spend_{ch}"
    mer_df[sc] = mer_df[ch].astype(float)
    spend_cols.append(sc)

# --- National weekly for Robyn track (Robyn-style national table) ---
df_nat = pd.read_csv(_data_abs)
df_nat = df_nat[df_nat["dmacode"].isin(regions)].copy()
df_nat = df_nat.sort_values(["dmacode", "weekid"])
for c in impression_cols:
    df_nat[c] = df_nat[c].fillna(0)
for c in control_cols + [target_col]:
    df_nat[c] = df_nat.groupby("dmacode")[c].ffill()
weeks_sorted = sorted(df_nat["weekid"].unique())
_weekid_to_date = {int(w): d for w, d in zip(_weeks_csv, dates_all)}
week_to_date = {int(w): _weekid_to_date[int(w)] for w in weeks_sorted}
df_nat["DATE"] = df_nat["weekid"].map(week_to_date)
_cal = sorted(df_nat["DATE"].dropna().unique())
_twc = len(_cal) - int(len(_cal) * HOLDOUT_RATIO)
_cut = _cal[_twc - 1]
_mask_tr = df_nat["DATE"] <= _cut
for c in control_cols + [target_col]:
    if df_nat[c].isna().any():
        m = df_nat.loc[_mask_tr, c].median()
        if pd.isna(m):
            m = float(df_nat.loc[_mask_tr, c].mean())
        df_nat[c] = df_nat[c].fillna(m)
agg_map = {c: "sum" for c in impression_cols + [target_col]}
agg_map.update({c: "mean" for c in control_cols})
nat = df_nat.groupby("DATE", as_index=False).agg(agg_map).sort_values("DATE").reset_index(drop=True)
rename_imp = dict(zip(impression_cols, [f"spend_{c}" for c in channel_short]))
nat = nat.rename(columns=rename_imp)
nat = nat.rename(columns={target_col: "dep_var"})
exposure_names = [f"imp_{c}" for c in channel_short]
for ch, ex in zip(channel_short, exposure_names):
    nat[ex] = nat[f"spend_{ch}"]
robyn_cols = ["DATE", "dep_var"] + [f"spend_{c}" for c in channel_short] + exposure_names + control_cols
robyn_national_df = nat[robyn_cols].copy()

# sklearn feature matrix for national Ridge (spend + imp + controls; no date)
feat_nat = [c for c in robyn_national_df.columns if c not in ("DATE", "dep_var")]
n_weeks_nat = len(robyn_national_df)
n_tr_nat = n_weeks_nat - int(n_weeks_nat * HOLDOUT_RATIO)
n_tr_nat = max(1, n_tr_nat)
rn_tr = robyn_national_df.iloc[:n_tr_nat]
rn_ho = robyn_national_df.iloc[n_tr_nat:]
Xr_tr = rn_tr[feat_nat].astype(float).values
yr_tr = rn_tr["dep_var"].astype(float).values
Xr_ho = rn_ho[feat_nat].astype(float).values
yr_ho = rn_ho["dep_var"].astype(float).values
ridge_nat = Ridge(alpha=1.0, random_state=RANDOM_SEED)
ridge_nat.fit(Xr_tr, yr_tr)
pred_tr_r = ridge_nat.predict(Xr_tr)
pred_ho_r = ridge_nat.predict(Xr_ho)

BENCHMARK_ROWS = []

def add_sklearn_row(name, scope, yt_tr, yp_tr, yt_ho, yp_ho, fit_s=None, extra=None):
    row = {
        "model": name,
        "scope": scope,
        "train_r2": float(r2_score(yt_tr, yp_tr)),
        "holdout_r2": float(r2_score(yt_ho, yp_ho)),
        "train_rmse": float(np.sqrt(mean_squared_error(yt_tr, yp_tr))),
        "holdout_rmse": float(np.sqrt(mean_squared_error(yt_ho, yp_ho))),
        "train_mae": float(mean_absolute_error(yt_tr, yp_tr)),
        "holdout_mae": float(mean_absolute_error(yt_ho, yp_ho)),
        "n_train": int(np.size(yt_tr)),
        "n_holdout": int(np.size(yt_ho)),
        "fit_s": fit_s,
    }
    naive = np.full_like(yt_ho, fill_value=float(np.mean(yt_tr)), dtype=float)
    row["holdout_rmse_naive_mean"] = float(np.sqrt(mean_squared_error(yt_ho, naive)))
    row["holdout_r2_naive_mean"] = float(r2_score(yt_ho, naive))
    _m = np.abs(np.asarray(yt_ho, dtype=float)) > 1e-9
    if _m.any():
        row["holdout_mape_pct"] = float(
            np.mean(np.abs((np.asarray(yt_ho)[_m] - np.asarray(yp_ho)[_m]) / np.asarray(yt_ho)[_m]))
            * 100.0
        )
    else:
        row["holdout_mape_pct"] = float("nan")
    if extra:
        row.update(extra)
    BENCHMARK_ROWS.append(row)

add_sklearn_row(
    "sklearn_Ridge_national_Robyn_input",
    "national_weekly",
    yr_tr,
    pred_tr_r,
    yr_ho,
    pred_ho_r,
    extra={"holdout_cut_date": str(_cut), "n_train_weeks": n_tr_nat, "n_holdout_weeks": len(rn_ho)},
)

print(
    f"Panel rows train/hold: {len(df_train)}/{len(df_holdout)} | "
    f"National weeks train/hold: {n_tr_nat}/{len(rn_ho)} | channels={len(channel_short)}"
)


 Loading Real MMM Data from: data/MMM Data.csv
    Loaded data shape: (20370, 23)
    Media channels (13): ['Channel 01', 'Channel 02', 'Channel 03', 'Channel 04', 'Channel 05', 'Channel 06', 'Channel 07', 'Channel 08', 'Channel 09', 'Channel 10 null', 'Channel 11', 'Channel 12', 'Channel 13']
    Control variables (7): ['Control 01', 'Control 02', 'Control 03', 'Control 04', 'Control 05', 'Control 06', 'Control 07']
    Data structure: 190 regions × 109 weeks
    Seasonality features will be added by UnifiedDataPipeline for proper train/holdout alignment
    Visits range: 0 - 15,778,575
    Real MMM data successfully loaded!
INFO - 
 Unified Data Pipeline - Time Series Split (Ratio-Based):
INFO -     Training: 96 actual weeks (weeks 1-96) - 88.1%
INFO -     Holdout: 13 actual weeks (weeks 97-109) - 11.9%
INFO -     Burn-in Padding: 6 weeks will be added to BOTH train and holdout
INFO -     Model sees: Train 102 weeks, Holdout 19 weeks
INFO -     Evaluation: Remove 6 padding weeks, eva

## DeepCausalMMM / DCM (dashboard-aligned panel)

Trains on **`UnifiedDataPipeline`** tensors (scaler fit on train only + leading **burn-in padding**), then **`ModelTrainer`** exactly like `dashboard_rmse_optimized.py`. **Train/holdout metrics** in the summary table use the same sklearn helpers as PyMC/Ridge on **original KPI scale** after **`inverse_transform_predictions(..., remove_padding=True)`**, i.e. evaluation on **all real weeks** (padding stripped), comparable to the PyMC panel rows in RMSE units.


In [ ]:
if not RUN_DCM:
    print("Skipping DeepCausalMMM / DCM (RUN_DCM=False)")
else:
    import torch

    from deepcausalmmm.core.config import get_default_config, update_config
    from deepcausalmmm.core.trainer import ModelTrainer

    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    dcm_tr_cfg = update_config(
        get_default_config(),
        {
            "n_epochs": int(DCM_N_EPOCHS),
            "burn_in_weeks": int(DCM_CONFIG.get("burn_in_weeks", 6)),
            "random_seed": RANDOM_SEED,
            "patience": int(DCM_EARLY_PATIENCE),
        },
    )

    # Dashboard path: dedicated pipeline (do not reuse unfitted `pipeline` from split-only cell)
    dcm_pipeline = UnifiedDataPipeline(DCM_CONFIG)
    train_tensors = dcm_pipeline.fit_and_transform_training(train_data)
    holdout_tensors = dcm_pipeline.transform_holdout(holdout_data)

    trainer = ModelTrainer(dcm_tr_cfg)
    n_media = int(train_tensors["X_media"].shape[2])
    n_control = int(train_tensors["X_control"].shape[2])
    n_regions = int(train_tensors["X_media"].shape[0])
    trainer.create_model(n_media, n_control, n_regions)
    trainer.create_optimizer_and_scheduler()

    t0 = time.perf_counter()
    trainer.train(
        train_tensors["X_media"],
        train_tensors["X_control"],
        train_tensors["R"],
        train_tensors["y"],
        holdout_tensors["X_media"],
        holdout_tensors["X_control"],
        holdout_tensors["R"],
        holdout_tensors["y"],
        pipeline=dcm_pipeline,
        verbose=True,
    )
    dcm_fit_s = time.perf_counter() - t0

    model = trainer.model
    model.eval()
    dev = trainer.device
    with torch.no_grad():
        pred_tr, _, _, _ = model(
            train_tensors["X_media"].to(dev),
            train_tensors["X_control"].to(dev),
            train_tensors["R"].to(dev),
        )
        pred_ho, _, _, _ = model(
            holdout_tensors["X_media"].to(dev),
            holdout_tensors["X_control"].to(dev),
            holdout_tensors["R"].to(dev),
        )

    def _orig_flat(y_scaled_torch):
        inv = dcm_pipeline.inverse_transform_predictions(
            y_scaled_torch.detach().cpu().float(), remove_padding=True
        )
        return np.asarray(inv.numpy(), dtype=np.float64).ravel()

    yt_tr = _orig_flat(train_tensors["y"])
    yp_tr = _orig_flat(pred_tr)
    yt_ho = _orig_flat(holdout_tensors["y"])
    yp_ho = _orig_flat(pred_ho)

    add_sklearn_row(
        "DeepCausalMMM_DCM_dashboard_path",
        "panel_geo_x_week",
        yt_tr,
        yp_tr,
        yt_ho,
        yp_ho,
        fit_s=dcm_fit_s,
        extra={
            "dcm_unified_pipeline": True,
            "dcm_inverse_transform_remove_padding": True,
            "dcm_burn_in_weeks": int(DCM_CONFIG.get("burn_in_weeks", 6)),
            "dcm_n_epochs_cap": int(DCM_N_EPOCHS),
            "dcm_early_stopping_patience": int(DCM_EARLY_PATIENCE),
        },
    )
    print(
        f"DCM fit done in {dcm_fit_s:.1f}s | "
        f"train_r2={r2_score(yt_tr, yp_tr):.4f} | holdout_r2={r2_score(yt_ho, yp_ho):.4f} | "
        f"holdout_rmse={np.sqrt(mean_squared_error(yt_ho, yp_ho)):.4g}"
    )


## PyMC-Marketing (panel)


In [11]:
if not RUN_PYMC:
    print("Skipping PyMC (RUN_PYMC=False)")
else:
    try:
        from pymc_extras.prior import Prior
        from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation
        from pymc_marketing.mmm.multidimensional import MMM
        from pymc_marketing.mmm.scaling import Scaling, VariableScaling
    except ImportError as e:
        print("PyMC-Marketing not installed:", e)
    else:
        def _align_panel_mu(mmm, X_df, pred, gcol, date_col="date"):
            pred = np.asarray(pred, dtype=float).ravel()
            dates = pd.to_datetime(pd.Index(sorted(X_df[date_col].unique())))
            geos = np.asarray(mmm.model_coords[gcol])
            n_d, n_g = len(dates), len(geos)
            wide = pred.reshape(n_d, n_g, order="C")
            mi = pd.MultiIndex.from_product([dates, geos], names=[date_col, gcol])
            long = pd.DataFrame({"_mu": wide.ravel(order="C")}, index=mi).reset_index()
            left = X_df[[date_col, gcol]].copy()
            left[date_col] = pd.to_datetime(left[date_col])
            long[gcol] = long[gcol].astype(left[gcol].dtype)
            long[date_col] = pd.to_datetime(long[date_col])
            out = left.merge(long, on=[date_col, gcol], how="left")
            return out["_mu"].to_numpy(dtype=float)

        adstock = GeometricAdstock(
            l_max=8,
            priors={"alpha": Prior("Beta", alpha=2, beta=5, dims="channel")},
        )
        saturation = LogisticSaturation(
            priors={
                "lam": Prior("Gamma", alpha=3, beta=1, dims="channel"),
                "beta": Prior("HalfNormal", sigma=2, dims="channel"),
            },
        )
        scaling = Scaling(
            target=VariableScaling(method="max", dims=()),
            channel=VariableScaling(method="max", dims=()),
        )
        mmm = MMM(
            date_column="date",
            target_column=target_col,
            channel_columns=channel_short,
            control_columns=control_cols,
            dims=(geo_col,),
            adstock=adstock,
            saturation=saturation,
            yearly_seasonality=2,
            scaling=scaling,
            model_config={"gamma_control": Prior("Normal", mu=0, sigma=2, dims="control")},
        )
        fit_kw = dict(
            chains=CHAINS,
            draws=DRAW,
            tune=TUNE,
            target_accept=TARGET_ACCEPT,
            random_seed=RANDOM_SEED,
            progressbar=True,
        )
        try:
            import numpyro  # noqa: F401
            fit_kw["nuts_sampler"] = "numpyro"
        except ImportError:
            pass
        t0 = time.perf_counter()
        mmm.fit(X_train, y_train, **fit_kw)
        fit_s = time.perf_counter() - t0

        y_pt = mmm.predict(X_train, extend_idata=False, include_last_observations=False)
        y_ph = mmm.predict(X_holdout, extend_idata=False, include_last_observations=True)
        y_pt = _align_panel_mu(mmm, X_train, y_pt, geo_col)
        y_ph = _align_panel_mu(mmm, X_holdout, y_ph, geo_col)
        ts = mmm.scalers._target
        if len(ts.dims) == 0:
            fac = float(np.asarray(ts).reshape(-1)[0])
            y_pt = y_pt * fac
            y_ph = y_ph * fac
        else:
            dn = list(ts.dims)[0]
            ser = pd.Series(
                np.asarray(ts.values).ravel(),
                index=pd.Index(ts.coords[dn].values, name=dn),
            )
            y_pt = y_pt * X_train[geo_col].map(ser).astype(float).to_numpy()
            y_ph = y_ph * X_holdout[geo_col].map(ser).astype(float).to_numpy()

        yt_tr = np.asarray(y_train, dtype=np.float64).ravel()
        yt_ho = np.asarray(y_holdout, dtype=np.float64).ravel()
        add_sklearn_row(
            "PyMC_Marketing_MMM",
            "panel_geo_x_week",
            yt_tr,
            np.asarray(y_pt, dtype=np.float64).ravel(),
            yt_ho,
            np.asarray(y_ph, dtype=np.float64).ravel(),
            fit_s=fit_s,
        )
        _pr = BENCHMARK_ROWS[-1]
        print(
            f"PyMC fit done in {fit_s:.1f}s | train_r2={_pr['train_r2']:.4f} (in-sample) | "
            f"holdout_r2={_pr['holdout_r2']:.4f} (OOS weeks) | "
            f"holdout_r2_naive_mean={_pr['holdout_r2_naive_mean']:.4f}"
        )


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [intercept_contribution, adstock_alpha, saturation_lam, saturation_beta, gamma_control, gamma_fourier, y_sigma]


Sampling 1 chain for 400 tune and 150 draw iterations (400 + 150 draws total) took 8330 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks


Sampling: [y]


Sampling: [y]


PyMC fit done in 5933.1s | train_r2=0.9938 (in-sample) | holdout_r2=0.9021 (OOS weeks) | holdout_r2_naive_mean=-0.0055


## Google Meridian (panel)


In [15]:
if not RUN_MERIDIAN:
    print("Skipping Meridian (RUN_MERIDIAN=False)")
else:
    try:
        from meridian import constants as mer_constants
        from meridian.data.data_frame_input_data_builder import DataFrameInputDataBuilder
        from meridian.model.model import Meridian
        from meridian.model.spec import ModelSpec
        from meridian.analysis.analyzer import Analyzer
    except ImportError as e:
        print("google-meridian not installed:", e)
    else:
        pop_df = mer_df.groupby("geo", as_index=False)["population"].first()
        builder = (
            DataFrameInputDataBuilder(kpi_type=mer_constants.NON_REVENUE)
            .with_kpi(mer_df, kpi_col="kpi", time_col="time", geo_col="geo")
            .with_revenue_per_kpi(mer_df, revenue_per_kpi_col="revenue_per_kpi", time_col="time", geo_col="geo")
            .with_population(pop_df, population_col="population", geo_col="geo")
            .with_controls(mer_df, control_cols=control_cols, time_col="time", geo_col="geo")
            .with_media(
                mer_df,
                media_cols=channel_short,
                media_spend_cols=spend_cols,
                media_channels=list(channel_short),
                time_col="time",
                geo_col="geo",
            )
        )
        input_data = builder.build()
        gdim, tdim = mer_constants.GEO, mer_constants.TIME
        n_g = int(input_data.kpi.sizes[gdim])
        n_t = int(input_data.kpi.sizes[tdim])
        holdout_start = n_t - int(holdout_weeks)
        holdout_id = np.zeros((n_g, n_t), dtype=bool)
        holdout_id[:, holdout_start:] = True
        mspec = ModelSpec(holdout_id=holdout_id)
        mer_m = Meridian(input_data, mspec)
        t0 = time.perf_counter()
        mer_m.sample_posterior(
            n_chains=N_CHAINS,
            n_adapt=N_ADAPT,
            n_burnin=N_BURNIN,
            n_keep=N_KEEP,
            seed=RANDOM_SEED,
        )
        fit_s = time.perf_counter() - t0
        an = Analyzer(model_context=mer_m.model_context, inference_data=mer_m.inference_data)
        acc = an.predictive_accuracy(use_kpi=True)
        mdf = acc.to_dataframe().reset_index()
        mdf.columns = [str(c).strip() for c in mdf.columns]
        gcol = "geo_granularity" if "geo_granularity" in mdf.columns else None
        escol = "evaluation_set" if "evaluation_set" in mdf.columns else None
        mcol = "metric" if "metric" in mdf.columns else ("Metric" if "Metric" in mdf.columns else None)
        vcol = "value" if "value" in mdf.columns else ("Value" if "Value" in mdf.columns else None)
        if not all([gcol, escol, mcol, vcol]):
            raise ValueError(f"Unexpected predictive_accuracy columns: {list(mdf.columns)}")

        def _eval_norm(x):
            return str(x).strip().lower()

        def _metric_key(series_row):
            m = str(series_row[mcol]).strip().lower().replace(" ", "_")
            return m

        geo_ok = mdf[gcol].astype(str).str.strip().str.lower() == "geo"
        sub = mdf[geo_ok & (mdf[escol].map(_eval_norm) == "test")]
        tr = mdf[geo_ok & (mdf[escol].map(_eval_norm) == "train")]

        row = {
            "model": "Meridian",
            "scope": "panel_geo_x_week",
            "train_r2": np.nan,
            "holdout_r2": np.nan,
            "train_rmse": np.nan,
            "holdout_rmse": np.nan,
            "train_mae": np.nan,
            "holdout_mae": np.nan,
            "n_train": int(len(df_train)),
            "n_holdout": int(len(df_holdout)),
            "fit_s": fit_s,
            "meridian_R2_geo_test": np.nan,
            "meridian_MAPE_geo_test": np.nan,
            "meridian_wMAPE_geo_test": np.nan,
            "holdout_rmse_naive_mean": float("nan"),
            "holdout_r2_naive_mean": float("nan"),
            "holdout_mape_pct": float("nan"),
        }

        r2_keys = ("r_squared", "r2", "r^2")
        for _, r in sub.iterrows():
            mk = _metric_key(r)
            if mk in r2_keys:
                v = float(r[vcol])
                row["meridian_R2_geo_test"] = v
                # Shared table column: Meridian "holdout" = vendor geo/Test R² (not sklearn on predictions).
                row["holdout_r2"] = v
            elif mk == "mape":
                row["meridian_MAPE_geo_test"] = float(r[vcol])
            elif mk == "wmape":
                row["meridian_wMAPE_geo_test"] = float(r[vcol])
        for _, r in tr.iterrows():
            mk = _metric_key(r)
            if mk in r2_keys:
                row["train_r2"] = float(r[vcol])
        if pd.notna(row["meridian_R2_geo_test"]):
            row["holdout_r2"] = float(row["meridian_R2_geo_test"])
        BENCHMARK_ROWS.append(row)
        print(f"Meridian sample_posterior done in {fit_s:.1f}s")
        display(mdf.head(20))


2026-03-22 16:18:28.951489: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator mcmc_retry_init/assert_equal_1/Assert/AssertGuard/Assert


Meridian sample_posterior done in 465.6s


,metric,geo_granularity,evaluation_set,value
0,R_Squared,geo,Train,9.966594e-01
1,R_Squared,geo,Test,-1.582513e+01
2,R_Squared,geo,All Data,-1.422516e+00
3,R_Squared,national,Train,9.997320e-01
4,R_Squared,national,Test,-6.899198e+04
5,R_Squared,national,All Data,-7.372182e+02
6,MAPE,geo,Train,inf
7,MAPE,geo,Test,inf
8,MAPE,geo,All Data,inf
9,MAPE,national,Train,1.299281e-03


## Optional: Robyn (`robynpy` + R)


In [13]:
if not RUN_ROBYN_GLMNET:
    print("Skipping Robyn glmnet (RUN_ROBYN_GLMNET=False)")
else:
    try:
        from robyn.robyn import Robyn
    except ImportError as e:
        print("robynpy not installed:", e)
    else:
        print("Robyn train_models can take several minutes...")
        # Minimal wiring — user should extend Channel, Hyperparameters, etc. per Robyn docs
        r = Robyn()
        # Example placeholder: actual API may differ by robynpy version
        print("Configure Robyn Channel/Hyperparameters in this cell for your environment.")


Skipping Robyn glmnet (RUN_ROBYN_GLMNET=False)


## Comparison table


In [16]:
summary = pd.DataFrame(BENCHMARK_ROWS)
# Flatten for display: sklearn block vs Meridian vendor cols
display(summary)

# Wide metric view (pivot-friendly)
cols_show = [
    "model",
    "scope",
    "train_r2",
    "holdout_r2",
    "train_rmse",
    "holdout_rmse",
    "train_mae",
    "holdout_mae",
    "holdout_mape_pct",
    "holdout_r2_naive_mean",
    "holdout_rmse_naive_mean",
    "meridian_R2_geo_test",
    "meridian_MAPE_geo_test",
    "meridian_wMAPE_geo_test",
    "fit_s",
]
cols_show = [c for c in cols_show if c in summary.columns]
display(summary[cols_show])


,model,scope,train_r2,holdout_r2,train_rmse,holdout_rmse,train_mae,holdout_mae,n_train,n_holdout,fit_s,holdout_rmse_naive_mean,holdout_r2_naive_mean,holdout_mape_pct,holdout_cut_date,n_train_weeks,n_holdout_weeks,meridian_R2_geo_test,meridian_MAPE_geo_test,meridian_wMAPE_geo_test
0,sklearn_Ridge_national_Robyn_input,national_weekly,0.855818,-12.429327,5.514719e+06,1.674411e+07,4.278349e+06,1.517053e+07,96,13,NaN,2.262715e+07,-23.523928,9.515496,2021-11-01 00:00:00,96.0,13.0,NaN,NaN,NaN
1,PyMC_Marketing_MMM,panel_geo_x_week,0.993801,0.902132,1.088599e+05,4.819179e+05,4.599751e+04,2.186550e+05,18240,2470,5933.138052,1.544695e+06,-0.005492,21.272982,NaN,NaN,NaN,NaN,NaN,NaN
2,Meridian,panel_geo_x_week,0.996659,NaN,NaN,NaN,NaN,NaN,18240,2470,455.552384,NaN,NaN,NaN,NaN,NaN,NaN,-15.825129,inf,5.631933
3,Meridian,panel_geo_x_week,0.996659,-15.825129,NaN,NaN,NaN,NaN,18240,2470,465.571070,NaN,NaN,NaN,NaN,NaN,NaN,-15.825129,inf,5.631933


,model,scope,train_r2,holdout_r2,train_rmse,holdout_rmse,train_mae,holdout_mae,holdout_mape_pct,holdout_r2_naive_mean,holdout_rmse_naive_mean,meridian_R2_geo_test,meridian_MAPE_geo_test,meridian_wMAPE_geo_test,fit_s
0,sklearn_Ridge_national_Robyn_input,national_weekly,0.855818,-12.429327,5.514719e+06,1.674411e+07,4.278349e+06,1.517053e+07,9.515496,-23.523928,2.262715e+07,NaN,NaN,NaN,NaN
1,PyMC_Marketing_MMM,panel_geo_x_week,0.993801,0.902132,1.088599e+05,4.819179e+05,4.599751e+04,2.186550e+05,21.272982,-0.005492,1.544695e+06,NaN,NaN,NaN,5933.138052
2,Meridian,panel_geo_x_week,0.996659,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15.825129,inf,5.631933,455.552384
3,Meridian,panel_geo_x_week,0.996659,-15.825129,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15.825129,inf,5.631933,465.571070
